# Forward validation — the Taylor–Green vortex

Before trusting the PINN on an inverse problem, I validate the Navier–Stokes residual code on a flow whose exact solution I know: the **Taylor–Green vortex**, an analytical solution of the 2D incompressible Navier–Stokes equations on $[0,2\pi]^2$ with periodic boundaries.

The network is trained on **physics only** — the initial condition, periodic boundary conditions, and the PDE residual. No interior velocity labels are used, so a low error against the analytical field genuinely means the physics engine works.

## Setup

The library code lives in `src/`. I fix all seeds and pick the GPU if available.

In [ ]:
import sys, torch
sys.path.insert(0, '.')  # so `src` imports when run from Week4/

from src.config import Config
from src.model import PINN
from src import data, pde, train, utils, viz

cfg = Config(n_layers=6, n_units=64, n_collocation=15000,
             adam_iters=2000, lbfgs_iters=1000, log_every=250)
utils.set_seed(cfg.seed)
device = utils.get_device(cfg.prefer_cuda)
generator = torch.Generator(device=device).manual_seed(cfg.seed)
device

## The stream-function trick

The network outputs $(\psi, p)$ and velocity is recovered as $u=\psi_y,\; v=-\psi_x$. Then $u_x+v_y = \psi_{yx}-\psi_{xy} \equiv 0$, so **mass conservation holds exactly by construction**. I check that numerically below — the divergence is at the level of floating-point round-off.

In [ ]:
model = PINN(cfg).to(device)
x, y, t = data.sample_collocation(cfg, device, generator)
for z in (x, y, t): z.requires_grad_(True)
u, v, _ = pde.predict_uvp(model, x, y, t)
div = (pde.grad(u, x) + pde.grad(v, y)).abs().max().item()
print(f'max |u_x + v_y| = {div:.2e}  (exact continuity by construction)')

## Train

Adam does the bulk of the optimization; L-BFGS then fine-tunes for the final accuracy. On a laptop RTX 4060 this takes a few minutes.

In [ ]:
history = train.train(model, cfg, device, generator)

## Validate against the analytical solution

I measure the relative $L^2$ error against the exact Taylor–Green fields across three time slices. Pressure is defined only up to an additive constant, so I mean-subtract it before comparing.

In [ ]:
eval_times = [cfg.t_min, 0.5*cfg.t_max, cfg.t_max]
for tt in eval_times:
    gx, gy, gt, shape = data.grid_eval_points(cfg, t=tt, n=100, device=device)
    for z in (gx, gy, gt): z.requires_grad_(True)
    up, vp, pp = pde.predict_uvp(model, gx, gy, gt)
    ut, vt, pt = data.tgv_fields(gx, gy, gt, cfg.nu)
    eu = utils.relative_l2(up, ut); ev = utils.relative_l2(vp, vt)
    ep = utils.relative_l2(pp-pp.mean(), pt-pt.mean())
    print(f't={tt:g}:  u={eu:.2%}  v={ev:.2%}  p={ep:.2%}')

## Fields

Predicted vs exact velocity, with the absolute-error map. The pressure panel is the interesting one — pressure was never supervised, it comes purely from the momentum equations.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import image as mpimg
for name in ['forward_u_t1.png', 'forward_p_t1.png']:
    plt.figure(figsize=(15, 4.2))
    plt.imshow(mpimg.imread(f'results/figures/{name}')); plt.axis('off'); plt.show()

These figures are regenerated end-to-end by `python run_forward.py`, which also writes the numbers to `results/metrics.json`.

With the engine validated, the [inverse reconstruction](02_inverse_reconstruction.ipynb) is where it gets interesting.